# 04 — Evaluation harness and baselines

Scored on the **validation window** (2022-07-01 → 2022-12-31). Main group = warm users (in train, ≥1 liked product in val); cold users reported separately. Products a user already bought in train are skipped.

In [1]:
import math
import sys

import pandas as pd

sys.path.insert(0, "../src")
from shoprank.evaluation import evaluate, ndcg_at_k, recall_at_k
from shoprank.retrieval.baselines import PopularRecommender, RandomRecommender

K = 10
T1 = pd.Timestamp("2022-07-01")

train = pd.read_parquet("../data/processed/train.parquet")
val = pd.read_parquet("../data/processed/val.parquet")
items = pd.read_parquet("../data/processed/items.parquet")
eval_users = pd.read_parquet("../data/processed/eval_users.parquet")

## Hand check

In [2]:
recs, liked = ["A", "B", "C"], {"B"}
print(f"Recall@3 = {recall_at_k(recs, liked, 3)}   NDCG@3 = {ndcg_at_k(recs, liked, 3):.4f}   1/log2(3) = {1 / math.log2(3):.4f}")
assert recall_at_k(recs, liked, 3) == 1.0
assert abs(ndcg_at_k(recs, liked, 3) - 1 / math.log2(3)) < 1e-12

Recall@3 = 1.0   NDCG@3 = 0.6309   1/log2(3) = 0.6309


## Ground truth and history

In [3]:
truth = val[val["liked"] == 1].groupby("user_id")["parent_asin"].agg(set).to_dict()
seen = train.groupby("user_id")["parent_asin"].agg(set).to_dict()

groups = eval_users[eval_users["split"] == "val"].groupby("group")["user_id"].agg(list).to_dict()
catalog_size = len(items)
print({g: len(u) for g, u in groups.items()}, " catalog:", catalog_size)
print(f"liked val products per warm user: {pd.Series([len(truth[u]) for u in groups['warm']]).describe().round(2).to_dict()}")

{'cold': 1495, 'warm': 4549}  catalog: 9999
liked val products per warm user: {'count': 4549.0, 'mean': 2.37, 'std': 2.71, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 3.0, 'max': 51.0}


## Baselines

In [4]:
models = {
    "Random": RandomRecommender(items["parent_asin"], seed=42),
    "Popular (all time)": PopularRecommender(train),
    "Trending (last 30 days)": PopularRecommender(train, days=30, cutoff=T1),
}

rows = []
for name, model in models.items():
    for group, users in groups.items():
        result = evaluate(model.recommend, users, truth, k=K,
                          seen=seen if group == "warm" else None, catalog_size=catalog_size)
        rows.append({"model": name, "group": group, **result})

results = pd.DataFrame(rows).set_index(["group", "model"]).sort_index()
results.round(4)

recall@10  precision@10  hit_rate@10  ndcg@10  \
group model                                                                    
cold  Popular (all time)          0.0110        0.0058       0.0468   0.0092   
      Random                      0.0012        0.0006       0.0060   0.0008   
      Trending (last 30 days)     0.0142        0.0073       0.0702   0.0112   
warm  Popular (all time)          0.0127        0.0032       0.0286   0.0071   
      Random                      0.0005        0.0002       0.0018   0.0004   
      Trending (last 30 days)     0.0251        0.0053       0.0517   0.0164   

                               coverage@10  users  
group model                                        
cold  Popular (all time)            0.0010   1495  
      Random                        0.7727   1495  
      Trending (last 30 days)       0.0010   1495  
warm  Popular (all time)            0.0016   4549  
      Random                        0.9878   4549  
      Trending (last 30 days)       0.0013   4549

In [5]:
trending = models["Trending (last 30 days)"]
recent_likes = train[(train["liked"] == 1) & (train["timestamp"] >= T1 - pd.Timedelta(days=30))]
print(f"liked interactions in the 30 days before T1: {len(recent_likes):,} on {recent_likes['parent_asin'].nunique():,} products")

titles = items.set_index("parent_asin")["title"]
pd.DataFrame({
    "popular (all time)": titles.reindex(models["Popular (all time)"].ranking[:K]).str[:60].to_list(),
    "trending (30 days)": titles.reindex(trending.ranking[:K]).str[:60].to_list(),
}, index=range(1, K + 1))

liked interactions in the 30 days before T1: 1,454 on 1,023 products


,popular (all time),trending (30 days)
1,amFilm Tempered Glass Screen Protector for Nin...,PlayStation DualSense Wireless Controller – Ga...
2,Nintendo Switch Pro Controller,Meta Quest 2 — Advanced All-In-One Virtual Rea...
3,DualShock 4 Wireless Controller for PlayStatio...,The Quarry - Xbox Series X
4,"SanDisk 128GB microSDXC-Card, Licensed for Nin...",Pokémon Legends: Arceus - Nintendo Switch
5,Super Smash Bros. Ultimate: Challenger Pack 2 ...,BOBOVR M2 Pro Battery Pack Head Strap Compatib...
6,The Legend of Zelda: Breath of the Wild Master...,"SanDisk 128GB microSDXC-Card, Licensed for Nin..."
7,Super Mario Odyssey - Nintendo Switch,"WolfLawS KM-3 Wireless Gaming Mouse, Rechargea..."
8,Mario Kart 8 Deluxe – Booster Course Pass - Ni...,KIWI design Head Strap with Halo Controller Pr...
9,$45 Nintendo eShop Gift Card [Digital Code],Kirby and the Forgotten Land - Nintendo Switch
10,PlayStation Plus: 12 Month Membership [Digital...,Turtle Beach Stealth 600 Gen 2 MAX Wireless Mu...


In [6]:
warm = results.loc["warm"]
print("| Model | Recall@10 | NDCG@10 | HitRate@10 | Precision@10 | Coverage |")
print("|---|---|---|---|---|---|")
for name, r in warm.loc[list(models)].iterrows():
    print(f"| {name} | {r['recall@10']:.4f} | {r['ndcg@10']:.4f} | {r['hit_rate@10']:.4f} | {r['precision@10']:.4f} | {r['coverage@10']:.2%} |")

| Model | Recall@10 | NDCG@10 | HitRate@10 | Precision@10 | Coverage |
|---|---|---|---|---|---|
| Random | 0.0005 | 0.0004 | 0.0018 | 0.0002 | 98.78% |
| Popular (all time) | 0.0127 | 0.0071 | 0.0286 | 0.0032 | 0.16% |
| Trending (last 30 days) | 0.0251 | 0.0164 | 0.0517 | 0.0053 | 0.13% |
